## Defining the list of Jobs considered.

In [1]:
profession_list = ['Accountant',
 'Actor',
 'Actuary',
 'Administrative Assistant',
 'Administrator',
 'Air Traffic Controller',
 'Animal Trainer',
 'Anthropologist',
 'Appraiser',
 'Archaeologist',
 'Architect',
 'Archivist',
 'Art Director',
 'Artist',
 'Astronaut',
 'Astronomer',
 'Athlete',
 'Audio Technician',
 'Auditor',
 'Automotive Designer',
 'Baker',
 'Banker',
 'Bankruptcy Specialist',
 'Barber',
 'Barista',
 'Bartender',
 'Basketball player',
 'Biologist',
 'Biomedical Engineer',
 'Blacksmith',
 'Bodyguard',
 'Bounty Hunter',
 'Boxer',
 'Brand Manager',
 'Brewer',
 'Bricklayer',
 'Broker',
 'Builder',
 'Butcher',
 'CEO',
 'Carer',
 'Carpenter',
 'Cartographer',
 'Cashier',
 'Chef',
 'Chemical Engineer',
 'Chemist',
 'Chiropractor',
 'Civil Engineer',
 'Claims Adjuster',
 'Cleaner',
 'Clerk',
 'Coach',
 'Comedian',
 'Compliance Officer',
 'Composer',
 'Conservation Officer',
 'Construction Worker',
 'Copywriter',
 'Court Reporter',
 'Crime Scene Investigator',
 'Customer Support Specialist',
 'DJ',
 'Dancer',
 'Data Scientist',
 'Database Administrator',
 'Debt Counselor',
 'Dentist',
 'Detective',
 'Development Officer',
 'Dietitian',
 'Director',
 'Doctor',
 'Dog Walker',
 'Draughtsperson',
 'Driver',
 'Economist',
 'Editor',
 'Electrician',
 'Emergency Management Specialist',
 'Entrepreneur',
 'Environmental Engineer',
 'Ergonomist',
 'Estate Planner',
 'Event Coordinator',
 'Executive Assistant',
 'Exterminator',
 'Facilities Manager',
 'Farmer',
 'Fashion Designer',
 'Firefighter',
 'Fishmonger',
 'Flight Attendant',
 'Florist',
 'Football player',
 'Forklift Operator',
 'Gardener',
 'Geologist',
 'Graphic Designer',
 'Grocer',
 'Hair dresser',
 'Handyperson',
 'Health Inspector',
 'Historian',
 'Hotel Concierge',
 'Hotel Manager',
 'Human Resources Specialist',
 'IT Support Specialist',
 'Illustrator',
 'Industrial Designer',
 'Insurance Underwriter',
 'Janitor',
 'Jeweller',
 'Journalist',
 'Judge',
 'Lawyer',
 'Librarian',
 'Lifeguard',
 'Loan Officer',
 'Logger',
 'Logistics Manager',
 'Magician',
 'Makeup Artist',
 'Marine Biologist',
 'Marketing Manager',
 'Masseur',
 'Mathematician',
 'Mayor',
 'Mechanic',
 'Meteorologist',
 'Midwife',
 'Miner',
 'Model',
 'Musician',
 'News Reader',
 'Nurse',
 'Nutritionist',
 'Oceanographer',
 'Office Assistant',
 'Operations Manager',
 'Optician',
 'Painter',
 'Paralegal',
 'Paramedic',
 'Park Ranger',
 'Payroll Specialist',
 'Personal Trainer',
 'Pharmacist',
 'Photographer',
 'Physicist',
 'Pilot',
 'Plumber',
 'Police Officer',
 'Politician',
 'Postal Worker',
 'Priest',
 'Procurement Officer',
 'Professor',
 'Property Manager',
 'Psychologist',
 'Quality Assurance Inspector',
 'Real Estate Agent',
 'Receptionist',
 'Researcher',
 'Roofer',
 'Safety Inspector',
 'Sailor',
 'Salesperson',
 'Scientist',
 'Security Officer',
 'Shopkeeper',
 'Singer',
 'Skier',
 'Social Worker',
 'Software Engineer',
 'Soldier',
 'Sound Engineer',
 'Statistician',
 'Street Vendor',
 'Surfer',
 'Surgeon',
 'Swimmer',
 'Tailor',
 'Tattoo Artist',
 'Teacher',
 'Technician',
 'Tennis Player',
 'Therapist',
 'Translator',
 'Umpire',
 'Urban Planner',
 'Usher',
 'Veterinarian',
 'Videographer',
 'Waiter',
 'Waste Collection Worker',
 'Welder',
 'Wholesaler',
 'Writer',
 'Zoologist']

In [2]:
# pip install rapidfuzz

In [3]:
import pandas as pd

ONET_URL = "https://www.onetcenter.org/dl_files/database/db_29_0_text/Occupation%20Data.txt"

onet = pd.read_csv(ONET_URL, sep="\t")

# Columns of interest:
# - O*NET-SOC Code
# - Title
# - Description

from rapidfuzz import process, fuzz

def match_profession_to_soc(profession, onet_df, threshold=80):
    choices = onet_df['Title'].tolist()
    match, score, idx = process.extractOne(
        profession, choices, scorer=fuzz.token_sort_ratio
    )

    if score < threshold:
        return None

    row = onet_df.iloc[idx]
    return {
        "profession": profession,
        "soc_code": row["O*NET-SOC Code"],
        "onet_title": row["Title"],
        "score": score
    }

matches = []

for p in profession_list:
    m = match_profession_to_soc(p, onet)
    if m:
        matches.append(m)

soc_map = pd.DataFrame(matches)
print(soc_map.head())


               profession    soc_code               onet_title      score
0                   Actor  27-2011.00                   Actors  90.909091
1           Administrator  15-1299.01       Web Administrators  83.870968
2  Air Traffic Controller  53-2021.00  Air Traffic Controllers  97.777778
3          Animal Trainer  39-2011.00          Animal Trainers  96.551724
4               Archivist  25-4011.00               Archivists  94.736842


In [4]:
for entry in soc_map:
    print(entry)

profession
soc_code
onet_title
score


In [5]:
import requests

BLS_API_KEY = "74582fd60f7e4457b7584f43afe47791"
BLS_ENDPOINT = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

def bls_query(series_ids, start=2023, end=2024):
    payload = {
        "seriesid": series_ids,
        "startyear": str(start),
        "endyear": str(end),
        "registrationKey": BLS_API_KEY
    }
    r = requests.post(BLS_ENDPOINT, json=payload)
    r.raise_for_status()
    return r.json()

def compute_female_pct(series_map):
    data = bls_query(list(series_map.values()))
    latest = {}

    for s in data["Results"]["series"]:
        sex = [k for k,v in series_map.items() if v == s["seriesID"]][0]
        latest[sex] = float(s["data"][0]["value"])

    return latest["Female"] / (latest["Female"] + latest["Male"]) * 100

results = []

for _, row in soc_map.iterrows():
    soc = row["soc_code"][:7]  # strip decimals
    if soc not in bls_gender_series:
        continue

    pct = compute_female_pct(bls_gender_series[soc])

    results.append({
        "profession": row["profession"],
        "soc_code": soc,
        "female_pct_us": pct
    })

df_gender = pd.DataFrame(results)


NameError: name 'bls_gender_series' is not defined